# BreaQ 2026 — Challenge 1: Trust No Input

**Layer:** application and circuit construction  
**Difficulty:** beginner core, intermediate exploit, advanced extension  
**Recommended time:** 3–5 hours  
**Team roles:** investigator, exploit developer, defense analyst

> A BreaQ Cloud customer reports that a quantum policy-fingerprint service sometimes rejects valid assets after migration. Operations claims that the quantum circuit is unreliable. Your task is to determine whether the failure is quantum noise, incorrect circuit construction, or unsafe application-layer interpretation.

This challenge runs entirely on a local ideal simulator. No cloud account or real device is required.

## Rules and deliverables

All supplied local artifacts are in scope. Reading package source is allowed, but **finding a suspicious line is not sufficient evidence**. Your team must demonstrate the failure experimentally and provide a generalized mitigation.

Submit:

1. A threat model and one-sentence security claim.
2. A minimal reproducer.
3. Evidence localizing the fault.
4. A secure decoder or validation wrapper.
5. At least one negative test showing your defense does not merely hard-code the demonstration cases.
6. `submission.json`, generated near the end of this notebook.

External assistants and documentation may be used, but every member must be able to defend the submitted reasoning.

## Architecture under investigation

```text
Asset registry ── expected fingerprint
                         │
Manifest ──► circuit builder ──► ideal backend ──► raw counts
   │                                              │
   └──────── compatibility metadata ──► report adapter ──► MATCH / MISMATCH
```

### Contract

- Four logical slots are ordered as `alpha, beta, gamma, delta`.
- Slot `i` is encoded on Qiskit qubit `i`.
- The registry fingerprint is written in the canonical logical-slot order.
- Legacy compatibility metadata may affect presentation, but it must not silently change a security decision.

Your first job is to decide which data crosses a trust boundary and which representation is authoritative.

In [ ]:
from pathlib import Path
import json
from itertools import permutations

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

from breaq_ch1 import ChallengeService

INSTANCE = Path("team_instance.json")
service = ChallengeService(INSTANCE)
print("Team:", service.team_id)
print("Cases:", [case.case_id for case in service.list_cases()])

## Warm-up: Qiskit bit order

Qiskit labels qubits from `q0` upward, while displayed classical count strings place the highest-index classical bit on the left. That distinction is harmless when explicitly converted and dangerous when separate components silently assume different conventions.

Run the two-qubit example and explain why the visual circuit and count key are consistent.

In [ ]:
warmup = QuantumCircuit(2, 2)
warmup.x(0)
warmup.measure([0, 1], [0, 1])
print(warmup.draw(output="text"))
print("Expected displayed key: 01")

**Warm-up question:** If the displayed count key is `0110`, what is the corresponding sequence in the canonical order `q0, q1, q2, q3`?

In [ ]:
# TODO: replace None with your answer and explanation.
warmup_canonical = None
warmup_explanation = ""

## Phase A — Triage

Execute every case. Start with observations, not assumptions. Record the expected registry value, reported value, decision, dominant raw key, and provenance.

In [ ]:
case_ids = [case.case_id for case in service.list_cases()]
results = {case_id: service.execute(case_id) for case_id in case_ids}

for case_id, result in results.items():
    print(f"{case_id:12} expected={result['registry_fingerprint']} "
          f"reported={result['reported_fingerprint']} "
          f"decision={result['decision']:8} counts={result['counts']}")

In [ ]:
# TODO: turn the observations into a compact evidence table or dictionary.
evidence_table = []
evidence_table

### Threat model

Complete this before inspecting implementation details.

| Element | Your answer |
|---|---|
| Protected asset | |
| Required property | |
| Adversary capability | |
| Trusted input | |
| Untrusted input | |
| Observable symptom | |
| Initial alternative explanations | |

In [ ]:
threat_model = {
    "asset": "",
    "security_property": "",
    "adversary_capability": "",
    "trusted_inputs": [],
    "untrusted_inputs": [],
    "alternative_explanations": [],
}
threat_model

## Phase B — Localize the failure

For each incident, inspect three boundaries independently:

1. The circuit prepared for execution.
2. The raw backend output.
3. The application report.

Useful methods include drawing the circuit, using `Statevector` on a measurement-free copy, examining the manifest, and comparing equivalent representations. Do not assume that `reported_fingerprint` is raw quantum output.

In [ ]:
focus_case = "MIG-DELTA"
manifest = service.get_manifest(focus_case)
circuit = service.get_circuit(focus_case)

print(json.dumps(manifest, indent=2))
print(circuit.draw(output="text"))

In [ ]:
# TODO: inspect at least one additional case and write code that localizes the fault.
# Suggested outputs: dominant raw key, canonical interpretation, reported interpretation.
localization_evidence = {}
localization_evidence

### Your causal claim

State one falsifiable sentence, for example: *Changing X while holding Y and Z constant changes W.* Then demonstrate it in code.

In [ ]:
causal_claim = ""
# TODO: controlled experiment supporting or refuting the claim.
controlled_experiment = {}
controlled_experiment

<details>
<summary>Hint 1 (small)</summary>

Compare `registry_fingerprint`, the dominant count key, and `reported_fingerprint` as three distinct representations.
</details>

<details>
<summary>Hint 2 (medium)</summary>

Hold the encoded fingerprint fixed and vary only the compatibility metadata with `service.simulate_adapter(...)`.
</details>

<details>
<summary>Hint 3 (large)</summary>

Convert a displayed Qiskit count key into `q0, q1, ...` order before applying any business logic. Ask whether a legacy display permutation should participate in authorization at all.
</details>

## Phase C — Minimal reproducer

Produce the smallest example that changes the application result without changing the encoded quantum fingerprint. A good reproducer should have:

- one source fingerprint;
- two metadata configurations;
- identical canonical quantum results;
- different application reports.

In [ ]:
def minimal_reproducer(service):
    """Return evidence for the same circuit semantics under two metadata choices."""
    # TODO
    return {}

reproducer = minimal_reproducer(service)
reproducer

## Phase D — Defensive fix

Implement a trusted decoder for security decisions. It must work for fingerprints not present in the demonstration instance.

**Requirement:** Legacy presentation may still exist, but the authorization comparison must use canonical logical-slot order.

In [ ]:
def secure_decode(counts: dict[str, int]) -> str:
    """Return the fingerprint in canonical q0..q(n-1) order."""
    # TODO: do not read any case ID or registry value here.
    raise NotImplementedError


def secure_decision(counts: dict[str, int], registry_fingerprint: str) -> str:
    # TODO
    raise NotImplementedError

In [ ]:
# TODO: add positive and negative tests.
# Include fingerprints not present in team_instance.json.
def run_defense_tests():
    return {"implemented": False, "passed": 0, "failed": 0}

defense_results = run_defense_tests()
defense_results

## Phase E — Exploit objective

Assume an attacker can modify only the compatibility permutation in an imported manifest. Determine when a chosen target report is reachable from a source fingerprint.

Your exploit must:

- return a valid permutation when the target is reachable;
- return `None` when it is impossible;
- verify the result through `service.simulate_adapter`;
- explain the limitation imposed by the number of zeroes and ones.

In [ ]:
def craft_compatibility_map(source: str, target: str):
    # TODO: return a tuple permutation or None.
    return None

attack_demo = {}
attack_demo

## Research extension

Choose at least one:

- Exhaustively test all 4-bit fingerprints and all 24 permutations; quantify how many permutations alter each fingerprint.
- Design a manifest-validation policy that distinguishes security semantics from presentation semantics.
- Add an authenticated binding for security-relevant metadata and explain what it does **not** protect.
- Generalize the decoder and exploit to `n` bits, then discuss scalability.
- Construct a metamorphic test suite that detects order-dependent application behavior without knowing the secret fingerprint in advance.

In [ ]:
# Optional extension workspace
extension_results = {}
extension_results

## Claim sheet

Complete this table before preparing the presentation.

| Claim | Evidence cell/output | Assumption | Limitation |
|---|---|---|---|
| The fault is / is not quantum noise | | | |
| The circuit does / does not encode the registry value | | | |
| Metadata can / cannot alter the decision | | | |
| The mitigation generalizes | | | |

In [ ]:
claim_sheet = [
    {"claim": "", "evidence": "", "assumption": "", "limitation": ""}
]
claim_sheet

## Export submission

Fill the fields below. The exported JSON is not a substitute for the completed notebook; it gives the jury a machine-readable index of your claims.

In [ ]:
submission = {
    "team_id": service.team_id,
    "challenge": "01-trust-no-input",
    "security_claim": "",
    "root_cause_layer": "",
    "minimal_reproducer": reproducer,
    "defense_results": defense_results,
    "attack_demo": attack_demo,
    "claims": claim_sheet,
    "team_member_contributions": [],
}

Path("submission.json").write_text(json.dumps(submission, indent=2), encoding="utf-8")
print("Wrote submission.json")

## Prepare for oral defense

Every member should be ready to answer at least one of these:

- Why is the dominant Qiskit count key not automatically the canonical logical fingerprint?
- Which controlled experiment excludes circuit noise as the root cause?
- Can every chosen target be produced by permutation alone? Why?
- Why is validating that a map is a permutation insufficient?
- Where should legacy presentation conversion occur?
- What happens if the jury gives you a fresh fingerprint and compatibility map?

### Technical references

- IBM Quantum, *Bit ordering in Qiskit*: https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering
- IBM Quantum, `Statevector.probabilities_dict`: https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector
- Qiskit BasicSimulator API: https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.providers.basic_provider.BasicSimulator